# Xenium pre-processing, clustering and cell-type annotation

Pre-processing and annotation pipeline for a 10x Xenium mouse-heart dataset comparing
wild-type (WT), base-edited (BE) and untreated mutant (R636Q / PBS) animals.

The notebook covers:

1. Conversion of raw Xenium output to per-sample `SpatialData` (Zarr) objects.
2. Concatenation of all samples into a single `AnnData` object.
3. QC metrics, cell and gene filtering.
4. Normalisation, scaling, PCA, UMAP and Leiden clustering.
5. Marker-based cluster annotation, plus sub-clustering of the lymphoid, myeloid and
   cardiomyocyte compartments.
6. Cell-type proportion tables and stacked bar charts per sample and per group.

## Annotation columns

Annotation is refined in stages; each stage adds a new `obs` column rather than
overwriting the previous one.

| Column | Contents |
| --- | --- |
| `celltype3` | `celltype2` with T and B cells and cleaned Ifgga4+ VCM cluster |
| `celltype4` | `celltype3` with T and B cells merged back into `Lymphoid` — the column used downstream |

## Inputs

| Input | Default location | Description |
| --- | --- | --- |
| Raw Xenium output | `data/raw/<sample>_resegment_largeCell/outs` | Per-sample Xenium runs after large-cell resegmentation |
| Sample metadata | `data/sample_ids_names_largecells_zarr.csv` | Semicolon-separated; columns `Sample_ID`, `Sample_name`, `Sample_type`, `Path` |

## Outputs

- Intermediate `AnnData` checkpoints in `results/processed/`
- Proportion tables (CSV) and figures (PDF) in `results/figures/`

All paths are configured in the **Configuration** cell and can be overridden with
environment variables, so nothing machine-specific is stored in the notebook.

## Reproducibility

**The cluster-to-cell-type maps in this notebook are tied to one specific run.**
Leiden cluster IDs are not stable across package versions or hardware, so re-running
the pipeline from scratch will very likely produce different cluster numbering. The
hardcoded maps (`"0": "Ifgga4+ VCMs"`, and so on) would then silently assign the wrong
labels. Random seeds are set explicitly below, but they only protect against run-to-run
variation within an identical environment.

If you re-run this pipeline, **re-derive the maps from the marker dot plots** rather
than trusting the ones committed here. To reproduce the published figures exactly,
start from the `AnnData` checkpoints instead of re-clustering.

## How to run

Run top to bottom. Section 4 (Zarr conversion) is a one-off step guarded by a flag and
is skipped by default. Every later section writes a checkpoint, so a fresh session can
resume part-way by reading the relevant file from `CHECKPOINTS`.

## 1. Setup

In [ ]:
import logging
import os
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_io as sio
from matplotlib.patches import Patch

### Configuration

Paths default to `data/` and `results/` next to the notebook. Point them elsewhere by
exporting these before launching Jupyter:

```bash
export XENIUM_DATA_DIR=/path/to/your/data
export XENIUM_RESULTS_DIR=/path/to/your/results
```

In [ ]:
# --- Input paths -----------------------------------------------------------
DATA_DIR = Path(os.environ.get("XENIUM_DATA_DIR", "data"))

# Raw Xenium output, one directory per sample (see RAW_SUBDIR_TEMPLATE below).
RAW_XENIUM_DIR = Path(os.environ.get("XENIUM_RAW_DIR", DATA_DIR / "raw"))
RAW_SUBDIR_TEMPLATE = "{sample_id}_resegment_largeCell"

# Semicolon-separated sample sheet: Sample_ID, Sample_name, Sample_type, Path.
SAMPLE_METADATA_CSV = Path(
    os.environ.get("XENIUM_SAMPLE_CSV", DATA_DIR / "sample_ids_names_largecells_zarr.csv")
)

# The sheet's `Path` column holds paths relative to the sheet's own location, not to
# the working directory, so this is prepended when the stores are opened. Change it if
# the Zarr stores have moved since the sheet was written.
ZARR_PATH_BASE = Path(
    os.environ.get("XENIUM_ZARR_PATH_BASE", SAMPLE_METADATA_CSV.parent)
)

# --- Output paths ----------------------------------------------------------
RESULTS_DIR = Path(os.environ.get("XENIUM_RESULTS_DIR", "results"))

ZARR_DIR = Path(os.environ.get("XENIUM_ZARR_DIR", DATA_DIR / "zarr"))
ZARR_NO_MORPHOLOGY_DIR = ZARR_DIR / "large_seg"
ZARR_WITH_MORPHOLOGY_DIR = ZARR_DIR / "large_seg_with_morphology"

PROCESSED_DIR = RESULTS_DIR / "processed"
FIG_DIR = RESULTS_DIR / "figures"

for path in (PROCESSED_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

# --- AnnData checkpoints ---------------------------------------------------
# Each pipeline stage writes one of these. To resume in a fresh session, read the
# checkpoint listed in the commented line at the end of the corresponding cell.
CHECKPOINTS = {
    "combined": PROCESSED_DIR / "largecells_all_withcoordinates.h5ad",
    "filtered": PROCESSED_DIR / "combined_Xenium_largecells_filtered.h5ad",
    "clustered": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_afterclustering.h5ad",
    "leiden": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_withleiden.h5ad",
    "annotated": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_annotated.h5ad",
    "annotated_ordered": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_annotated_ordered.h5ad",
    "with_lymphoid_myeloid": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_withTcells.h5ad",
    "reannotated_ifgga4": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_reannotatedIfgga4.h5ad",
    "final": PROCESSED_DIR / "combined_Xenium_largecells_noWT3_final_celltype4.h5ad",
    "lymphoid_subset": PROCESSED_DIR / "Xenium_lymphoid_subset_BcellsTcells.h5ad",
    "myeloid_subset": PROCESSED_DIR / "Xenium_myeloid_subset.h5ad",
}

# Seeds are passed explicitly to every stochastic step. These match the scanpy
# defaults, so setting them does not change existing results; they are stated so the
# intent is visible and so a future scanpy default change cannot silently alter output.
RANDOM_SEED = 0

print("Raw Xenium  :", RAW_XENIUM_DIR)
print("Sample sheet:", SAMPLE_METADATA_CSV)
print("Checkpoints :", PROCESSED_DIR.resolve())
print("Figures     :", FIG_DIR.resolve())

In [ ]:
# TrueType fonts so text stays editable in Illustrator/Inkscape.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# fontTools is very chatty when subsetting fonts for PDF export.
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

# Directory used by scanpy's own `save=` argument.
sc.settings.figdir = FIG_DIR
sc.settings.set_figure_params(dpi_save=300, transparent=True)

## 2. Study design and constants

Twelve samples across three groups: four WT, four base-edited (BE) and four untreated
R636Q mutants (labelled `PBS`).

`WT_rep3` was excluded after the initial combine because the section was cut at the
wrong orientation, so it does not appear in `SAMPLE_ORDER` below. Whether it appears in
`SAMPLE_IDS` depends on whether it was converted to Zarr in the first place — check
`SAMPLE_METADATA_CSV`, which is what actually drives the combine step.

In [ ]:
# Samples converted to SpatialData objects in section 4.
SAMPLE_IDS = [
    "WT_rep1", "WT_rep2", "WT_rep4", "WT_rep5",
    "BE_rep1", "BE_rep2", "BE_rep3", "BE_rep4",
    "PBS_rep1", "PBS_rep2", "PBS_rep3", "PBS_rep4",
]

# Experimental groups, in figure order.
GROUP_ORDER = ["WT", "BE", "R636Q"]

# Samples retained for analysis, in figure order (WT_rep3 excluded).
SAMPLE_ORDER = [
    "WT_rep1", "WT_rep2", "WT_rep4", "WT_rep5",
    "BE_rep1", "BE_rep2", "BE_rep3", "BE_rep4",
    "PBS_rep1", "PBS_rep2", "PBS_rep3", "PBS_rep4",
]

# QC thresholds.
MIN_COUNTS_PER_CELL = 50
MIN_CELLS_PER_GENE = 3

In [ ]:
# Cell-type orders and matching palettes. The palette changes between annotation
# stages because the number of categories changes (T/B cells split out, then merged).

CELLTYPE_ORDER_BROAD = [
    "VCMs", "Ifgga4+ VCMs", "Stressed VCMs", "Myh7+ VCMs", "FBs",
    "Vasculature ECs", "Endocardial ECs", "Epicardium", "Myeloid", "Lymphoid",
    "Pericytes", "SMCs", "NCs", "ACMs",
]

CELLTYPE_ORDER_WITH_TB = [
    "VCMs", "Ifgga4+ VCMs", "Stressed VCMs", "Myh7+ VCMs", "FBs",
    "Vasculature ECs", "Endocardial ECs", "Epicardium", "Myeloid",
    "T cells", "B cells", "Pericytes", "SMCs", "NCs", "ACMs",
]

# 14 colours, for CELLTYPE_ORDER_BROAD at the first annotation stage (`celltype`).
PALETTE_BROAD = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB",
    "#44AA99", "#117733", "#999933", "#AA4499", "#7BB4C3", "#CC6677", "#332288",
]

# 15 colours, for CELLTYPE_ORDER_WITH_TB (`celltype2`, `celltype3`).
PALETTE_WITH_TB = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB",
    "#44AA99", "#117733", "#999933", "#EE7733", "#AA4499", "#7BB4C3", "#CC6677",
    "#4B3FA3",
]

# 14 colours, for CELLTYPE_ORDER_BROAD at the final stage (`celltype4`).
PALETTE_FINAL = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB",
    "#44AA99", "#117733", "#999933", "#AA4499", "#7BB4C3", "#CC6677", "#4B3FA3",
]

In [ ]:
# Custom probes targeting Rbm20-dependent splicing isoforms and edited alleles. These
# are excluded from highly variable genes when sub-clustering non-cardiomyocyte
# compartments, where they carry no meaningful signal but dominate the variance.
RBM20_ISOFORM_PROBES = [
    "1300002E11Rik-SE-neg", "2210408F21Rik-SE-pos", "Aak1-SE-neg",
    "Abhd14a-SE-pos", "Ank3-SE-pos", "Arhgap10-MXE-neg", "Arhgap10-MXE-pos",
    "Aste1-ASS-pos", "Camk2d-SE-pos", "Camk2d_flanking-exons",
    "Camk2d_isoformA", "Camk2d_isoformB", "Carnmt1-MXE-pos",
    "Ccdc125-MXE-neg", "Ccdc17-SE-pos", "Cdk5rap2-MXE-neg",
    "Col13a1-SE-neg", "Commd1-SE-pos", "Csde1-SE-neg", "Dcun1d2-SE-pos",
    "Dnm1l-SE-neg", "Etl4-SE-pos", "Far1-MXE-pos", "Fbf1-MXE-neg",
    "Gm16565-SE-pos", "Gm27252-SE-neg", "Gm46430-SE-neg", "H2-T10-SE-pos",
    "Hdnr-SE-neg", "Hmgn3-SE-neg", "Hnrnpdl-SE-pos", "Hyi-SE-pos",
    "Immt_alt-exon", "Immt_alt-exon-junction", "Immt_flanking-exons",
    "Kcng2-SE-pos", "Ldb3-SE-neg", "Ldb3_MUT-alt-exon", "Ldb3_WT-alt-exon",
    "Ldb3_flanking-exons", "Lrrfip2-SE-neg", "Lyplal1-SE-neg",
    "Med15-MXE-pos", "Mical1-ASS-pos", "Mlip-SE-pos", "Mov10-SE-neg",
    "Mutyh-SE-pos", "Nup210-ASS-neg", "Oas1c-SE-neg",
    "Pdlim5_double-junction", "Pdlim5_flanking-exons", "Phldb1-SE-neg",
    "Pkd2l2-SE-neg", "Polr1b-SE-pos", "Postn-SE-pos", "Ppa2-SE-pos",
    "Pstk-SE-neg", "Rbm20-P635L", "Rbm20-R636Q", "Rbm20-R636Q_edited",
    "Rbm20-WT", "Rbm20_binder-sequence", "Rbm6-SE-pos", "Rnf44-ASS-pos",
    "Ryr2_flanking-exons", "Sdccag8-SE-neg", "Sel1l-ASS-neg",
    "Sema3b-SE-pos", "Slc25a37-RI-neg", "Smox-MXE-pos", "Sorbs1-RI-pos",
    "Spata6-SE-neg", "Svil-SE-neg", "Tor1aip1-SE-pos", "Tpcn2-SE-neg",
    "Tpm2-ASS-pos", "Tpm2-MXE-neg", "Tpm2-SE-pos", "Tpm2_MT", "Tpm2_WT",
    "Trmt1-RI-pos", "Ttc3-SE-neg", "Ttn-SE-pos", "Ttn_N2A", "Ttn_N2B",
    "Ttn_WT-N2B", "Ttn_alt-exon", "Ttn_flanking-exons", "Ube2f-SE-neg",
    "Wiz-SE-neg", "Zdhhc3-SE-neg", "Zfp120-SE-pos", "Zfp644-SE-neg",
]

print(f"{len(RBM20_ISOFORM_PROBES)} isoform probes defined")

## 3. Helper functions

In [ ]:
def drop_isoform_probes_from_hvg(adata, probes=RBM20_ISOFORM_PROBES):
    """Mark the Rbm20 isoform probes as not highly variable, in place.

    Prints how many were matched, so a probe name that no longer exists in the panel
    (e.g. after a rename) fails loudly instead of being silently ignored by the
    intersection.
    """
    present = adata.var_names.intersection(probes)
    missing = sorted(set(probes) - set(adata.var_names))

    adata.var.loc[present, "highly_variable"] = False

    print(f"Excluded {len(present)}/{len(probes)} isoform probes from HVGs")
    if missing:
        print(f"  not present in the panel: {len(missing)} -> {missing[:5]}...")

    return present


def transfer_labels(target, source, source_col, out_col, base_col):
    """Copy finer sub-cluster labels from a subset object back onto the full object.

    Starts from ``base_col``, overwrites the cells shared with ``source`` using
    ``source.obs[source_col]``, and stores the result in ``target.obs[out_col]`` as an
    unordered categorical. ``base_col`` and ``out_col`` may be the same column, which
    layers a second transfer on top of a first.
    """
    labels = target.obs[base_col].astype(str).copy()
    shared_cells = target.obs_names.intersection(source.obs_names)
    labels.loc[shared_cells] = source.obs.loc[shared_cells, source_col].astype(str)

    target.obs[out_col] = pd.Categorical(labels)

    print(f"{out_col}: updated {len(shared_cells)} cells from {source_col}")
    return shared_cells

In [ ]:
def celltype_proportions(adata, cell_col, sample_col="name", group_col="type"):
    """Cell-type proportions per sample, and their mean per experimental group.

    All three columns must already be ordered categoricals; their category order sets
    the row and column order of the returned tables.

    Returns
    -------
    (props_sample, group_means)
        ``props_sample`` rows sum to 1 within each sample. ``group_means`` is the
        unweighted mean across the samples in each group.
    """
    obs = adata.obs[[cell_col, sample_col, group_col]].copy()

    celltype_order = list(obs[cell_col].cat.categories)
    sample_order = list(obs[sample_col].cat.categories)
    group_order = list(obs[group_col].cat.categories)

    props_sample = (
        pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
        .reindex(index=sample_order, columns=celltype_order)
        .fillna(0)
    )

    sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]

    tmp = props_sample.copy()
    tmp[group_col] = sample2group.reindex(sample_order).values

    group_means = (
        tmp.groupby(group_col, observed=True)[celltype_order]
        .mean()
        .reindex(index=group_order)
    )

    return props_sample, group_means


def plot_stacked_proportions(
    props,
    palette,
    out_file,
    ylabel="Cell type proportion",
    legend_title="celltype",
    figsize=(8, 5),
    bar_width=0.9,
):
    """Stacked proportion bar chart, one bar per row of ``props``.

    Segments are stacked in reversed column order so the top-to-bottom order of each
    bar matches the top-to-bottom order of the legend.
    """
    celltype_order = list(props.columns)
    color_dict = dict(zip(celltype_order, palette))
    plot_order = celltype_order[::-1]

    fig, ax = plt.subplots(figsize=figsize)

    props[plot_order].plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[color_dict[ct] for ct in plot_order],
        width=bar_width,
    )

    ax.legend(
        handles=[Patch(facecolor=color_dict[ct], label=ct) for ct in celltype_order],
        title=legend_title,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    ax.grid(False)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)

    plt.tight_layout()

    out_file = Path(out_file)
    fig.savefig(out_file, dpi=300, bbox_inches="tight", transparent=True)
    print(f"Saved: {out_file}")

    plt.show()
    plt.close(fig)


def set_plotting_orders(adata, cell_col, celltype_order):
    """Apply the canonical categorical orders used by every proportion figure."""
    adata.obs[cell_col] = pd.Categorical(
        adata.obs[cell_col], categories=celltype_order, ordered=True
    )
    adata.obs["type"] = pd.Categorical(
        adata.obs["type"], categories=GROUP_ORDER, ordered=True
    )
    adata.obs["name"] = pd.Categorical(
        adata.obs["name"], categories=SAMPLE_ORDER, ordered=True
    )

## 4. Convert raw Xenium output to SpatialData objects

One-off step: reads each resegmented Xenium run and writes it as a Zarr-backed
`SpatialData` object. Guarded by a flag because it is slow and only needs to run once.

Two variants are written:

- **Without morphology images** (`large_seg/`) — used by the combine step below. The
  `morphology_focus` images are dropped because they are not needed here and make the
  stores much larger.
- **With morphology images** (`large_seg_with_morphology/`) — required by the ROI
  visualisation notebook, which draws cell masks on the DAPI images.

In [ ]:
RUN_ZARR_CONVERSION = False

if RUN_ZARR_CONVERSION:
    ZARR_NO_MORPHOLOGY_DIR.mkdir(parents=True, exist_ok=True)
    ZARR_WITH_MORPHOLOGY_DIR.mkdir(parents=True, exist_ok=True)

    for sample_id in SAMPLE_IDS:
        print(f"Processing {sample_id}...")

        xenium_path = (
            RAW_XENIUM_DIR / RAW_SUBDIR_TEMPLATE.format(sample_id=sample_id) / "outs"
        )
        sdata = sio.xenium(xenium_path)

        # Keep the full object, morphology images included, for the ROI notebook.
        with_morph_path = ZARR_WITH_MORPHOLOGY_DIR / f"{sample_id}_with_morphology.zarr"
        sdata.write(with_morph_path)

        # Then drop the images and write the lighter object used downstream here.
        if "morphology_focus" in sdata:
            del sdata["morphology_focus"]

        no_morph_path = ZARR_NO_MORPHOLOGY_DIR / f"{sample_id}.zarr"
        sdata.write(no_morph_path)

        print(f"  -> {with_morph_path}")
        print(f"  -> {no_morph_path}")
else:
    print("Skipped: set RUN_ZARR_CONVERSION = True to regenerate the Zarr stores.")

## 5. Combine samples into a single AnnData object

Driven by the sample sheet rather than `SAMPLE_IDS`, so the sheet is the single source
of truth for which samples enter the analysis. Cell IDs are prefixed with the sample ID
to keep `obs_names` unique across samples.

In [ ]:
meta_samples = pd.read_csv(SAMPLE_METADATA_CSV, sep=";")
meta_samples.head()

In [ ]:
adatas = []

for _, row in meta_samples.iterrows():
    sample = row["Sample_ID"]

    sdata = sd.read_zarr(ZARR_PATH_BASE / row["Path"])
    adata = sdata["table"]

    # Keep the original within-sample cell IDs before making obs_names unique.
    adata.obs["cell_id_orig"] = adata.obs["cell_id"]
    adata.obs["id"] = sample

    # Sample-level annotation from the sheet.
    adata.obs["type"] = row["Sample_type"]
    adata.obs["name"] = row["Sample_name"]

    adata.obs_names = [f"{sample}_{cid}" for cid in adata.obs["cell_id_orig"]]

    adatas.append(adata)

combined_adata = ad.concat(adatas, axis=0, index_unique=None)
combined_adata

In [ ]:
print(combined_adata.obs.columns.tolist())
print(combined_adata.obsm.keys())
print(combined_adata.obsm["spatial"][:5])

combined_adata.obs["id"].value_counts()

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["combined"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["combined"])

## 6. QC metrics, filtering and the counts layer

`percent_top` is set explicitly because the scanpy default reaches 500 genes, which
raises an `IndexError` on a panel this small. These values measure what fraction of a
cell's counts is taken up by its most-expressed genes: a cell where the top 10 genes
hold 90% of the counts has low complexity and is likely to be problematic.

In [ ]:
sc.pp.calculate_qc_metrics(
    combined_adata,
    percent_top=(10, 20, 50, 150, 200),
    inplace=True,
    log1p=True,
)

In [ ]:
# jitter is reduced from the default 0.4, which is too dense to read at this cell count.
sc.pl.violin(
    combined_adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0.1,
    groupby="name",
    multi_panel=True,
)

In [ ]:
# Counts distribution, with the filtering threshold marked.
plt.hist(combined_adata.obs["total_counts"], range=(0, 2000), bins=100)
plt.axvline(x=MIN_COUNTS_PER_CELL, color="r", linestyle="--")
plt.xlabel("Total counts per cell")
plt.ylabel("Number of cells")
plt.show()

In [ ]:
sc.pl.violin(combined_adata, keys="pct_counts_in_top_50_genes", stripplot=False)

In [ ]:
# Keep the raw counts before filtering, in case they are needed later.
combined_adata.layers["counts"] = combined_adata.X.copy()

sc.pp.filter_cells(combined_adata, min_counts=MIN_COUNTS_PER_CELL)
sc.pp.filter_genes(combined_adata, min_cells=MIN_CELLS_PER_GENE)

# Refresh the layer so it matches the filtered object.
combined_adata.layers["counts"] = combined_adata.X.copy()

combined_adata

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["filtered"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["filtered"])

### Exclude WT_rep3

`WT_rep3` was sectioned at the wrong orientation and is dropped from all downstream
analysis.

In [ ]:
print("Before:", sorted(combined_adata.obs["name"].unique()))

combined_adata = combined_adata[combined_adata.obs["name"] != "WT_rep3"].copy()

print("After :", sorted(combined_adata.obs["name"].unique()))

## 7. Normalisation, scaling and PCA

In [ ]:
# HVGs with the seurat_v3 flavour, which expects raw counts.
sc.pp.highly_variable_genes(
    combined_adata, flavor="seurat_v3", n_top_genes=200, layer="counts"
)

In [ ]:
sc.pp.normalize_total(combined_adata)
sc.pp.log1p(combined_adata)
combined_adata.layers["lognorm"] = combined_adata.X.copy()

In [ ]:
# Scaling without zero-centring, to keep the matrix sparse at this dataset size.
sc.pp.scale(combined_adata, zero_center=False, max_value=10)
sc.pp.pca(combined_adata, n_comps=30, random_state=RANDOM_SEED)

In [ ]:
# Used to choose how many PCs to carry into the neighbourhood graph.
sc.pl.pca_variance_ratio(combined_adata, n_pcs=30, log=True)

## 8. UMAP embedding and Leiden clustering

In [ ]:
sc.pp.neighbors(combined_adata, n_neighbors=100, n_pcs=25, random_state=RANDOM_SEED)
sc.tl.umap(combined_adata, min_dist=0.05, spread=2, random_state=RANDOM_SEED)

In [ ]:
sc.pl.umap(combined_adata, color="type")

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["clustered"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["clustered"])

In [ ]:
sc.tl.leiden(
    combined_adata,
    flavor="igraph",
    n_iterations=-1,
    resolution=1,
    key_added="leiden_r1",
    random_state=RANDOM_SEED,
)

In [ ]:
sc.pl.umap(combined_adata, color="leiden_r1")

In [ ]:
# F830016B08Rik is the Ifgga4 probe.
sc.pl.umap(combined_adata, color="F830016B08Rik")

## 9. Cluster annotation

Clusters are annotated from their top differentially expressed genes.

**The map below is specific to one clustering run.** If you re-ran section 8, inspect
the dot plot first and rebuild the map — the cluster numbers will not match.

In [ ]:
sc.tl.rank_genes_groups(combined_adata, groupby="leiden_r1", method="wilcoxon")
sc.tl.dendrogram(combined_adata, groupby="leiden_r1")

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    combined_adata, groupby="leiden_r1", standard_scale="var", n_genes=10
)

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["leiden"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["leiden"])

In [ ]:
# Leiden cluster -> broad cell type. Comments record the observations behind the calls.
cluster_map = {
    "0": "Ifgga4+ VCMs",
    "1": "VCMs",
    "2": "VCMs",
    "3": "VCMs",            # also Myh7b positive
    "4": "Stressed VCMs",   # irregular shape; cleaned up in section 13
    "5": "Vasculature ECs",
    "6": "Myeloid",
    "7": "FBs",
    "8": "Stressed VCMs",
    "9": "VCMs",
    "10": "FBs",
    "11": "Myh7+ VCMs",
    "12": "NCs",
    "13": "VCMs",
    "14": "Pericytes",
    "15": "Stressed VCMs",
    "16": "SMCs",
    "17": "Lymphoid",
    "18": "Vasculature ECs",
    "19": "Endocardial ECs",
    "20": "Vasculature ECs",
    "21": "Epicardium",     # mesothelial cells
    "22": "FBs",
    "23": "ACMs",
}

combined_adata.obs["celltype"] = combined_adata.obs["leiden_r1"].map(cluster_map)

combined_adata.obs["celltype"].value_counts()

In [ ]:
sc.pl.umap(combined_adata, color="celltype")

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["annotated"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["annotated"])

In [ ]:
combined_adata.obs["celltype"] = pd.Categorical(
    combined_adata.obs["celltype"], categories=CELLTYPE_ORDER_BROAD, ordered=True
)

sc.pl.umap(
    combined_adata, color="celltype", palette=PALETTE_BROAD, save="UMAP_noWT3.pdf"
)

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["annotated_ordered"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["annotated_ordered"])

## 10. Marker visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
sc.set_figure_params(dpi=300)

for ax, (group, title) in zip(
    axes, [("R636Q", "Mutant"), ("WT", "WT"), ("BE", "Base-edited")]
):
    sc.pl.umap(
        combined_adata[combined_adata.obs["type"] == group],
        color="leiden_r1",
        size=2,
        ax=ax,
        title=title,
        legend_loc=None,
        show=False,
        frameon=False,
    )

plt.tight_layout()
plt.show()

In [ ]:
marker_genes = [
    "Ttn", "F830016B08Rik", "Ankrd1", "Myh7", "Pdgfra", "Cdh5", "Vwf",
    "Muc16", "F13a1", "Skap1", "Rgs5", "Tagln", "Chl1", "Nppa",
]

sc.set_figure_params(dpi=300, dpi_save=300, transparent=True)

sc.pl.stacked_violin(
    combined_adata,
    marker_genes,
    groupby="celltype",
    swap_axes=False,
    dendrogram=False,
    figsize=(8, 4),
    cmap="viridis",
    save="markers_violin.pdf",
)

In [ ]:
# Lymphoid and myeloid markers, to check the compartments worth sub-clustering.
sc.pl.umap(combined_adata, color=["Cd74", "Bank1", "Ms4a1", "Cd247", "Skap1"])

## 11. Lymphoid sub-clustering (T cells and B cells)

The lymphoid compartment is re-clustered from raw counts. Rbm20 isoform probes are
excluded from the HVGs because they are only informative in cardiomyocytes.

In [ ]:
LC = combined_adata[combined_adata.obs["celltype"] == "Lymphoid"].copy()

# Restart from raw counts for the sub-clustering.
LC.X = LC.layers["counts"].copy()

LC

In [ ]:
sc.pp.highly_variable_genes(LC, flavor="seurat_v3", layer="counts")
drop_isoform_probes_from_hvg(LC)

In [ ]:
sc.pp.normalize_total(LC)
sc.pp.log1p(LC)
LC.layers["lognorm"] = LC.X.copy()

sc.pp.scale(LC, max_value=10)
sc.pp.pca(LC, n_comps=30, random_state=RANDOM_SEED)

In [ ]:
sc.pl.pca_variance_ratio(LC, n_pcs=30, log=True)

In [ ]:
sc.pp.neighbors(LC, n_pcs=20, random_state=RANDOM_SEED)
sc.tl.umap(LC, random_state=RANDOM_SEED)

sc.pl.umap(LC, color="type")

In [ ]:
sc.tl.leiden(
    LC, flavor="igraph", resolution=0.2, key_added="leiden_r02", random_state=RANDOM_SEED
)

sc.pl.umap(LC, color="leiden_r02")

In [ ]:
sc.pl.umap(
    LC,
    color=[
        "Cd74", "Bank1", "Ms4a1", "Cd247", "Skap1", "Il2ra", "Cd4",
        "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86",
    ],
)

In [ ]:
sc.tl.rank_genes_groups(LC, method="wilcoxon", groupby="leiden_r02")
sc.pl.rank_genes_groups_dotplot(
    LC, groupby="leiden_r02", standard_scale="var", n_genes=15
)

In [ ]:
# Specific to this sub-clustering run - re-derive if section 11 was re-run.
cluster_map = {
    "0": "T cells",
    "1": "T cells",
    "2": "T cells",
    "3": "B cells",
    "4": "Ccr1+ Myeloid",
}

LC.obs["celltype_LC"] = LC.obs["leiden_r02"].map(cluster_map)

sc.pl.umap(LC, color="celltype_LC")

In [ ]:
LC.write_h5ad(CHECKPOINTS["lymphoid_subset"])

# Resume from here:
# LC = sc.read_h5ad(CHECKPOINTS["lymphoid_subset"])

## 12. Myeloid sub-clustering

Same procedure as the lymphoid compartment.

The resulting labels are written to `celltype1` alongside the lymphoid ones. **This
column is exploratory and is not carried forward** — `celltype2` in section 13 is built
from `celltype` plus the lymphoid labels only, so the myeloid sub-clusters do not appear
in any downstream figure.

In [ ]:
MC = combined_adata[combined_adata.obs["celltype"] == "Myeloid"].copy()
MC.X = MC.layers["counts"].copy()

sc.pp.highly_variable_genes(MC, flavor="seurat_v3", layer="counts")
drop_isoform_probes_from_hvg(MC)

In [ ]:
sc.pp.normalize_total(MC)
sc.pp.log1p(MC)
MC.layers["lognorm"] = MC.X.copy()

sc.pp.scale(MC, max_value=10)
sc.pp.pca(MC, n_comps=30, random_state=RANDOM_SEED)

sc.pl.pca_variance_ratio(MC, n_pcs=30, log=True)

In [ ]:
sc.pp.neighbors(MC, n_pcs=20, random_state=RANDOM_SEED)
sc.tl.umap(MC, random_state=RANDOM_SEED)

sc.pl.umap(MC, color="type")

In [ ]:
sc.tl.leiden(
    MC, flavor="igraph", resolution=0.3, key_added="leidenr03", random_state=RANDOM_SEED
)

sc.pl.umap(MC, color="leidenr03")

In [ ]:
sc.pl.umap(
    MC,
    color=[
        "Cd74", "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86",
        "Cd163", "S100a1", "Adgre1", "Ms4a7", "Lair1", "Ptprc", "Tnnt2", "Myl2",
    ],
)

In [ ]:
sc.tl.rank_genes_groups(MC, method="wilcoxon", groupby="leidenr03")
sc.pl.rank_genes_groups_dotplot(
    MC, groupby="leidenr03", standard_scale="var", n_genes=10
)

In [ ]:
genes_of_interest = [
    "Cd74", "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86", "Cd163",
    "S100a1", "Adgre1", "Ms4a7", "Lair1", "Ptprc", "Col1a2", "Rbm20", "Tnnt2", "Myl2",
]

sc.pl.rank_genes_groups_dotplot(
    MC, var_names=genes_of_interest, groupby="leidenr03", standard_scale="var"
)

In [ ]:
# Specific to this sub-clustering run - re-derive if section 12 was re-run.
cluster_map = {
    "0": "MCs",
    "1": "MCs",
    "2": "MC-like ECs",
    "3": "MC-like FBs",
    "4": "SMCs",
    "5": "MCs",
}

MC.obs["celltype_MC"] = MC.obs["leidenr03"].map(cluster_map)

sc.pl.umap(MC, color="celltype_MC")

In [ ]:
MC.write_h5ad(CHECKPOINTS["myeloid_subset"])

# Resume from here:
# MC = sc.read_h5ad(CHECKPOINTS["myeloid_subset"])

In [ ]:
# celltype1: broad annotation + lymphoid + myeloid sub-clusters (exploratory only).
transfer_labels(combined_adata, LC, "celltype_LC", out_col="celltype1", base_col="celltype")
transfer_labels(combined_adata, MC, "celltype_MC", out_col="celltype1", base_col="celltype1")

sc.pl.umap(combined_adata, color="celltype1")

## 13. Cell-type proportions with T and B cells

`celltype2` is the annotation used for the first set of proportion figures: the broad
annotation with T and B cells split out, but without the myeloid sub-clusters. The
`Ccr1+ Myeloid` cluster picked up by the lymphoid sub-clustering is folded back into
`Myeloid`.

In [ ]:
transfer_labels(combined_adata, LC, "celltype_LC", out_col="celltype2", base_col="celltype")

# The lymphoid sub-clustering split off a myeloid cluster; fold it back in.
combined_adata.obs["celltype2"] = combined_adata.obs["celltype2"].replace(
    {"Ccr1+ Myeloid": "Myeloid"}
)

# Drop the stale colour mapping so scanpy rebuilds it for the new categories.
combined_adata.uns.pop("celltype2_colors", None)

sc.pl.umap(combined_adata, color="celltype2")

In [ ]:
set_plotting_orders(combined_adata, "celltype2", CELLTYPE_ORDER_WITH_TB)

sc.pl.umap(
    combined_adata,
    color="celltype2",
    palette=PALETTE_WITH_TB,
    save="UMAP_noWT3_withT_Bcells.pdf",
)

In [ ]:
props_sample, group_means = celltype_proportions(combined_adata, "celltype2")

props_sample.to_csv(FIG_DIR / "samplenoWT3_Xenium_celltypes.csv")
group_means.to_csv(FIG_DIR / "group_means_Xenium_celltypes.csv")

group_means

In [ ]:
plot_stacked_proportions(
    props_sample,
    palette=PALETTE_WITH_TB,
    out_file=FIG_DIR / "celltype_proportions_per_sample.pdf",
    ylabel="Cell type proportion",
)

plot_stacked_proportions(
    group_means,
    palette=PALETTE_WITH_TB,
    out_file=FIG_DIR / "celltype_mean_proportions_per_group.pdf",
    ylabel="Mean cell type proportion",
    figsize=(4.5, 5),
    bar_width=0.8,
)

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["with_lymphoid_myeloid"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["with_lymphoid_myeloid"])

## 14. Cardiomyocyte sub-type cleanup

The Ifgga4+ VCM cluster clearly contains Ifgga4-negative cells based on the feature
plot, so the VCM compartment is re-clustered and re-assigned. The result is stored as
`celltype3`.

In [ ]:
VCM_CELLTYPES = ["VCMs", "Ifgga4+ VCMs", "Stressed VCMs", "Myh7+ VCMs"]

VCMs_all = combined_adata[combined_adata.obs["celltype"].isin(VCM_CELLTYPES)].copy()

# Marker panel per VCM subtype, used for the grouped dot plot.
genes_by_group = {
    "VCMs": ["Myl2", "Ttn_N2B", "Ttn_flanking-exons"],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons"],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon"],
    "Myh7+ VCMs": ["Myh7"],
}

sc.pl.rank_genes_groups_dotplot(
    VCMs_all,
    groupby="celltype",
    var_names=genes_by_group,
    standard_scale="var",
    dendrogram=False,
    save="dotplot_mygenes_grouped_VCMsspatial.pdf",
)

### 14.1 Re-cluster the Ifgga4+ VCMs

In [ ]:
Ifgga4 = combined_adata[combined_adata.obs["celltype"] == "Ifgga4+ VCMs"].copy()

sc.tl.leiden(
    Ifgga4, flavor="igraph", resolution=0.5, key_added="leidenr03",
    random_state=RANDOM_SEED,
)

sc.pl.umap(Ifgga4, color=["leidenr03", "F830016B08Rik"])

In [ ]:
sc.tl.rank_genes_groups(Ifgga4, method="wilcoxon", groupby="leidenr03")
sc.pl.rank_genes_groups_dotplot(
    Ifgga4, groupby="leidenr03", standard_scale="var", n_genes=10
)

In [ ]:
# Specific to this sub-clustering run - re-derive if the cell above was re-run.
cluster_map = {
    "0": "Ifgga4+ VCMs",
    "1": "Vasculature ECs",
    "2": "Ifgga4+ VCMs",
    "3": "VCMs",
}

Ifgga4.obs["celltypeIfgga4"] = Ifgga4.obs["leidenr03"].map(cluster_map)

sc.pl.umap(Ifgga4, color="celltypeIfgga4")

In [ ]:
transfer_labels(
    combined_adata, Ifgga4, "celltypeIfgga4", out_col="celltype3", base_col="celltype2"
)

sc.pl.umap(combined_adata, color="celltype3")

### 14.2 Exploratory: stressed VCMs

Inspected but not re-assigned; the clustering did not support splitting this compartment
further.

In [ ]:
Stressed = combined_adata[combined_adata.obs["celltype"] == "Stressed VCMs"].copy()

sc.tl.leiden(
    Stressed, flavor="igraph", resolution=0.8, key_added="leidenr08",
    random_state=RANDOM_SEED,
)

sc.pl.umap(Stressed, color=["leidenr08", "F830016B08Rik", "Cdh5"])

In [ ]:
sc.tl.rank_genes_groups(Stressed, method="wilcoxon", groupby="leidenr08")
sc.pl.rank_genes_groups_dotplot(
    Stressed, groupby="leidenr08", standard_scale="var", n_genes=15
)

### 14.3 Re-cluster the remaining VCMs

In [ ]:
VCMs = combined_adata[combined_adata.obs["celltype"] == "VCMs"].copy()

sc.tl.leiden(
    VCMs, flavor="igraph", resolution=0.5, key_added="leidenr05",
    random_state=RANDOM_SEED,
)

sc.pl.umap(VCMs, color=["leidenr05", "F830016B08Rik", "Cdh5", "type"])

In [ ]:
sc.tl.rank_genes_groups(VCMs, method="wilcoxon", groupby="leidenr05")
sc.pl.rank_genes_groups_dotplot(
    VCMs, groupby="leidenr05", standard_scale="var", n_genes=15
)

In [ ]:
# Specific to this sub-clustering run - re-derive if the cell above was re-run.
cluster_map = {
    "0": "VCMs",
    "1": "Ifgga4+ VCMs",
    "2": "VCMs",
    "3": "VCMs",
    "4": "VCMs",
    "5": "VCMs",
    "6": "VCMs",
}

VCMs.obs["celltypeIfgga4"] = VCMs.obs["leidenr05"].map(cluster_map)

# Layered on top of the Ifgga4 transfer above, so base and target are both celltype3.
transfer_labels(
    combined_adata, VCMs, "celltypeIfgga4", out_col="celltype3", base_col="celltype3"
)

sc.pl.umap(combined_adata, color=["celltype3", "F830016B08Rik"])

In [ ]:
set_plotting_orders(combined_adata, "celltype3", CELLTYPE_ORDER_WITH_TB)

sc.pl.umap(
    combined_adata,
    color="celltype3",
    palette=PALETTE_WITH_TB,
    save="UMAP_noWT3_withT_Bcells_reannotatedIfgga4.pdf",
)

In [ ]:
# Spot-check the re-assignment in space on one section.
BE_rep1 = combined_adata[combined_adata.obs["name"] == "BE_rep1"].copy()

sc.pl.spatial(
    BE_rep1,
    color=["celltype3"],
    groups=["Ifgga4+ VCMs"],
    na_color="lightgrey",
    spot_size=25,
)

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["reannotated_ifgga4"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["reannotated_ifgga4"])

### 14.4 Proportions using the re-annotated cell types

In [ ]:
props_sample, group_means = celltype_proportions(combined_adata, "celltype3")

props_sample.to_csv(FIG_DIR / "samplenoWT3_Xenium_celltypes_Ifgga4reannotated.csv")
group_means.to_csv(FIG_DIR / "group_means_Xenium_celltypes_Ifgga4reannotated.csv")

group_means

In [ ]:
plot_stacked_proportions(
    props_sample,
    palette=PALETTE_WITH_TB,
    out_file=FIG_DIR / "celltype_proportions_per_sample_Ifgga4reannotated.pdf",
    ylabel="Cell type proportion",
)

plot_stacked_proportions(
    group_means,
    palette=PALETTE_WITH_TB,
    out_file=FIG_DIR / "celltype_mean_proportions_per_group_Ifgga4reannotated.pdf",
    ylabel="Mean cell type proportion",
    figsize=(4.5, 5),
    bar_width=0.8,
)

In [ ]:
# Grouped VCM marker dot plot, now on the re-annotated compartment.
VCMs_all = combined_adata[combined_adata.obs["celltype"].isin(VCM_CELLTYPES)].copy()

sc.pl.rank_genes_groups_dotplot(
    VCMs_all,
    groupby="celltype",
    var_names=genes_by_group,
    standard_scale="var",
    dendrogram=False,
    save="dotplot_mygenes_grouped_VCMsspatial_Ifgga4_reannotated.pdf",
)

## 15. Final annotation (`celltype4`)

T and B cells are merged back into a single `Lymphoid` category. `celltype4` is the
annotation consumed by the downstream ROI visualisation notebook.

In [ ]:
combined_adata.obs["celltype4"] = (
    combined_adata.obs["celltype3"]
    .astype(str)
    .replace({"T cells": "Lymphoid", "B cells": "Lymphoid"})
    .astype("category")
)

set_plotting_orders(combined_adata, "celltype4", CELLTYPE_ORDER_BROAD)

combined_adata.obs["celltype4"].value_counts()

In [ ]:
sc.pl.umap(
    combined_adata,
    color="celltype4",
    palette=PALETTE_FINAL,
    save="UMAP_noWT3_withT_Bcells_reannotatedIfgga4_noTBcells.pdf",
)

In [ ]:
combined_adata.write_h5ad(CHECKPOINTS["final"])

# Resume from here:
# combined_adata = sc.read_h5ad(CHECKPOINTS["final"])

In [ ]:
props_sample, group_means = celltype_proportions(combined_adata, "celltype4")

props_sample.to_csv(
    FIG_DIR / "samplenoWT3_Xenium_celltypes_Ifgga4reannotated_noTBjustLymphoid.csv"
)
group_means.to_csv(
    FIG_DIR / "group_means_Xenium_celltypes_Ifgga4reannotated_noTBjustLymphoid.csv"
)

group_means

In [ ]:
plot_stacked_proportions(
    props_sample,
    palette=PALETTE_FINAL,
    out_file=(
        FIG_DIR / "celltype_proportions_per_sample_Ifgga4reannotated_noTBjustLymphoid.pdf"
    ),
    ylabel="Cell type proportion",
)

plot_stacked_proportions(
    group_means,
    palette=PALETTE_FINAL,
    out_file=(
        FIG_DIR
        / "celltype_mean_proportions_per_group_Ifgga4reannotated_noTBjustLymphoid.pdf"
    ),
    ylabel="Mean cell type proportion",
    figsize=(4.5, 5),
    bar_width=0.8,
)

## 16. Cardiomyocyte sub-type composition

Cardiomyocyte proportions rescaled so the VCM subtypes sum to 1 within each sample,
making the shift between subtypes visible without the non-cardiomyocyte compartments
dominating the bars.

In [ ]:
# Colours are taken from the head of PALETTE_FINAL so the subtypes keep the same
# colours they have in the full celltype4 UMAP, where they are the first categories.
cm_celltypes = [ct for ct in VCM_CELLTYPES if ct in props_sample.columns]

props_cm_sample = props_sample[cm_celltypes].copy()
props_cm_sample = props_cm_sample.div(props_cm_sample.sum(axis=1), axis=0).fillna(0)

plot_stacked_proportions(
    props_cm_sample,
    palette=PALETTE_FINAL[: len(cm_celltypes)],
    out_file=FIG_DIR / "cardiomyocyte_subtype_proportions_per_sample.pdf",
    ylabel="Proportion of cardiomyocytes",
    legend_title="Cardiomyocyte subtype",
)

In [ ]:
# Spatial check of the final VCM subtypes on one section.
BE_rep1 = combined_adata[combined_adata.obs["name"] == "BE_rep1"].copy()

for groups in (VCM_CELLTYPES, ["Stressed VCMs", "Myh7+ VCMs"], ["Myh7+ VCMs"]):
    sc.pl.spatial(
        BE_rep1,
        color=["celltype3"],
        groups=groups,
        na_color="lightgrey",
        spot_size=25,
    )

## 17. Handover to the ROI visualisation notebook

> **Incomplete — this step is missing from the repository.**

The ROI notebook (`Xenium03_ROI_visualisation.ipynb`) expects an object that this
notebook does not currently produce. Two things differ:

1. **Filename.** It reads `..._ReannotatedIfgga4_newcolours.h5ad`; the final checkpoint
   here is `CHECKPOINTS["final"]`.
2. **Cell-type labels.** It expects `Basal VCMs`, `IFN-associated VCMs`,
   `Remodelled VCMs`; `celltype4` here uses `VCMs`, `Ifgga4+ VCMs`, `Myh7+ VCMs`.

There is evidently a renaming and re-colouring step between the two notebooks that is
not in the repository. The draft mapping below matches the two label sets on the
assumption that the rename was purely cosmetic — **verify it against whatever actually
produced the `_newcolours` file before relying on it**, and add the colour assignment
that step performed. Until then the cell is disabled.

In [ ]:
RUN_NEWCOLOURS_STEP = False

# DRAFT - verify against the original renaming step before enabling.
CELLTYPE4_RENAME = {
    "VCMs": "Basal VCMs",
    "Ifgga4+ VCMs": "IFN-associated VCMs",
    "Myh7+ VCMs": "Remodelled VCMs",
    # "Stressed VCMs", "Myeloid" and "Lymphoid" are unchanged.
}

if RUN_NEWCOLOURS_STEP:
    combined_adata.obs["celltype4"] = (
        combined_adata.obs["celltype4"].astype(str).replace(CELLTYPE4_RENAME)
    )

    renamed_order = [CELLTYPE4_RENAME.get(ct, ct) for ct in CELLTYPE_ORDER_BROAD]
    set_plotting_orders(combined_adata, "celltype4", renamed_order)

    # TODO: apply whatever colour assignment the original "_newcolours" step used.

    out_path = PROCESSED_DIR / (
        "combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_"
        "ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad"
    )
    combined_adata.write_h5ad(out_path)
    print(f"Saved: {out_path}")
else:
    print("Skipped: see the notes above before enabling RUN_NEWCOLOURS_STEP.")